#**MovieLens 1M Analysis: Colaborative Filtering**  
**Group L03 - Team 07**

# 1. Data Loading

In [ ]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import pickle
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [ ]:
# if running on gg colab
import gdown
gdown.download_folder('https://drive.google.com/drive/folders/1HkzgKNjgJ6H9edInSBvFj60gc52y4equ?usp=drive_link', quiet=False, use_cookies=False, output="./")

Retrieving folder contents


Processing file 1C_MwVj5tpshYnbHuDTNoYPN6m0lO0CV0 movies.dat
Processing file 1RaGo4q8ofqSaVYOExl70ARP5BvdwAK-Z ratings.dat
Processing file 1AH9EDZHQKyRvzOS74YbzdhNXPFa7lLaz README
Processing file 1XPeMfuH0dxJIy2zIk7s6n1QZ5iVWPgEy users.dat


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1C_MwVj5tpshYnbHuDTNoYPN6m0lO0CV0
To: /content/ml-1m/movies.dat
100%|██████████| 175k/175k [00:00<00:00, 41.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1RaGo4q8ofqSaVYOExl70ARP5BvdwAK-Z
To: /content/ml-1m/ratings.dat
100%|██████████| 25.6M/25.6M [00:00<00:00, 103MB/s] 
Downloading...
From: https://drive.google.com/uc?id=1AH9EDZHQKyRvzOS74YbzdhNXPFa7lLaz
To: /content/ml-1m/README
100%|██████████| 5.75k/5.75k [00:00<00:00, 7.74MB/s]
Downloading...
From: https://drive.google.com/uc?id=1XPeMfuH0dxJIy2zIk7s6n1QZ5iVWPgEy
To: /content/ml-1m/users.dat
100%|██████████| 140k/140k [00:00<00:00, 59.6MB/s]
Download completed


['./ml-1m/movies.dat',
 './ml-1m/ratings.dat',
 './ml-1m/README',
 './ml-1m/users.dat']

In [ ]:
DATA_PATH = './ml-1m/'
ratings = pd.read_csv(DATA_PATH + 'ratings.dat', sep='::', engine='python', names=['UserID', 'MovieID', 'Rating', 'Timestamp'])
users = pd.read_csv(DATA_PATH + 'users.dat', sep='::', engine='python', names=['UserID', 'Gender', 'Age', 'Occupation', 'Zipcode'])
movies = pd.read_csv(DATA_PATH + 'movies.dat', sep='::', engine='python', names=['MovieID', 'Title', 'Genres'], encoding='latin-1')
print(f"Ratings: {len(ratings):,} | Users: {len(users):,} | Movies: {len(movies):,}")

Ratings: 1,000,209 | Users: 6,040 | Movies: 3,883


In [ ]:
df = ratings.merge(users, on='UserID').merge(movies, on='MovieID')
df['GenreList'] = df['Genres'].str.split('|')
df['Datetime'] = pd.to_datetime(df['Timestamp'], unit='s')
print(f"Merged: {df.shape}")
df.head()

Merged: (1000209, 12)


,UserID,MovieID,Rating,Timestamp,Gender,Age,Occupation,Zipcode,Title,Genres,GenreList,Datetime
0,1,1193,5,978300760,F,1,10,48067,One Flew Over the Cuckoo's Nest (1975),Drama,[Drama],2000-12-31 22:12:40
1,1,661,3,978302109,F,1,10,48067,James and the Giant Peach (1996),Animation|Children's|Musical,"[Animation, Children's, Musical]",2000-12-31 22:35:09
2,1,914,3,978301968,F,1,10,48067,My Fair Lady (1964),Musical|Romance,"[Musical, Romance]",2000-12-31 22:32:48
3,1,3408,4,978300275,F,1,10,48067,Erin Brockovich (2000),Drama,[Drama],2000-12-31 22:04:35
4,1,2355,5,978824291,F,1,10,48067,"Bug's Life, A (1998)",Animation|Children's|Comedy,"[Animation, Children's, Comedy]",2001-01-06 23:38:11


# 2. Colaborative Filtering Pipeline

In [ ]:
user_counts  = df.groupby('UserID')['Rating'].count()
movie_counts = df.groupby('MovieID')['Rating'].count()
active_users  = user_counts[user_counts  >= 50].index
active_movies = movie_counts[movie_counts >= 50].index

df_filtered = df[df['UserID'].isin(active_users) & df['MovieID'].isin(active_movies)].copy()
print(f"Filtered: {df_filtered['UserID'].nunique():,} users | "
      f"{df_filtered['MovieID'].nunique():,} movies | "
      f"{len(df_filtered):,} ratings")

Filtered: 4,297 users | 2,514 movies | 922,127 ratings


In [ ]:
train_df, test_df = train_test_split(df_filtered, test_size=0.2, random_state=42)
print(f"Train: {len(train_df):,}  |  Test: {len(test_df):,}")

Train: 737,701  |  Test: 184,426


In [ ]:
def build_user_item_matrix(data):
    """Return pivot table with NaN for unrated cells."""
    return data.pivot_table(index='UserID', columns='MovieID', values='Rating')

train_matrix = build_user_item_matrix(train_df)
print(f"User-Item matrix: {train_matrix.shape}  |  "
      f"Density: {train_matrix.notna().sum().sum() / train_matrix.size * 100:.2f}%")
user_means = train_matrix.mean(axis=1)
train_matrix_norm = train_matrix.subtract(user_means, axis=0).fillna(0)
train_matrix_norm

User-Item matrix: (4297, 2514)  |  Density: 6.83%


MovieID      1     2         3         4         5         6         7     \
UserID                                                                      
1        0.825000   0.0  0.000000  0.000000  0.000000  0.000000  0.000000   
2        0.000000   0.0  0.000000  0.000000  0.000000  0.000000  0.000000   
3        0.000000   0.0  0.000000  0.000000  0.000000  0.000000  0.000000   
5        0.000000   0.0  0.000000  0.000000  0.000000  0.000000  0.000000   
6        0.000000   0.0  0.000000  0.000000  0.000000  0.000000  0.000000   
...           ...   ...       ...       ...       ...       ...       ...   
6035     1.354545   0.0 -1.645455 -0.645455 -1.645455  0.000000  0.354545   
6036     0.000000   0.0  0.000000 -1.306333  0.000000 -0.306333  0.000000   
6037     0.000000   0.0  0.000000  0.000000  0.000000  0.000000  0.000000   
6039     0.000000   0.0  0.000000  0.000000  0.000000  0.000000  0.000000   
6040     0.000000   0.0  0.000000  0.000000  0.000000  0.000000  0.000000   

MovieID  8     9     10        11    12    13    14    15        16    \
UserID                                                                  
1         0.0   0.0   0.0  0.000000   0.0   0.0   0.0   0.0  0.000000   
2         0.0   0.0   0.0  0.000000   0.0   0.0   0.0   0.0  0.000000   
3         0.0   0.0   0.0  0.000000   0.0   0.0   0.0   0.0  0.000000   
5         0.0   0.0   0.0  0.000000   0.0   0.0   0.0   0.0 -0.176471   
6         0.0   0.0   0.0  0.000000   0.0   0.0   0.0   0.0  0.000000   
...       ...   ...   ...       ...   ...   ...   ...   ...       ...   
6035      0.0   0.0   0.0  0.000000   0.0   0.0   0.0   0.0  0.000000   
6036      0.0   0.0   0.0 -0.306333   0.0   0.0   0.0   0.0 -0.306333   
6037      0.0   0.0   0.0  0.000000   0.0   0.0   0.0   0.0  0.000000   
6039      0.0   0.0   0.0  0.000000   0.0   0.0   0.0   0.0  0.000000   
6040      0.0   0.0   0.0  0.000000   0.0   0.0   0.0   0.0  0.000000   

MovieID      17    18    19    20        21    22    23        24        25    \
UserID                                                                          
1        0.000000   0.0   0.0   0.0  0.000000   0.0   0.0  0.000000  0.000000   
2        0.000000   0.0   0.0   0.0  0.000000   0.0   0.0  0.000000  0.000000   
3        0.000000   0.0   0.0   0.0  0.000000   0.0   0.0  0.000000  0.000000   
5        0.000000   0.0   0.0   0.0  0.000000   0.0   0.0  0.000000  0.000000   
6        0.000000   0.0   0.0   0.0  0.000000   0.0   0.0  0.000000  0.000000   
...           ...   ...   ...   ...       ...   ...   ...       ...       ...   
6035     0.000000   0.0   0.0   0.0  0.000000   0.0   0.0  0.000000 -1.645455   
6036     0.693667   0.0   0.0   0.0 -0.306333   0.0   0.0 -1.306333  0.693667   
6037     0.263804   0.0   0.0   0.0  0.000000   0.0   0.0  0.000000  0.000000   
6039     0.000000   0.0   0.0   0.0  0.000000   0.0   0.0  0.000000  0.000000   
6040    -0.552124   0.0   0.0   0.0  0.000000   0.0   0.0  0.000000 -0.552124   

MovieID      26    27        28        29        30    31        32    \
UserID                                                                  
1        0.000000   0.0  0.000000  0.000000  0.000000   0.0  0.000000   
2        0.000000   0.0  0.000000  0.000000  0.000000   0.0  0.000000   
3        0.000000   0.0  0.000000  0.000000  0.000000   0.0  0.000000   
5        0.000000   0.0  0.000000  0.000000  0.000000   0.0  0.823529   
6        0.000000   0.0  0.000000  0.000000  0.000000   0.0  0.000000   
...           ...   ...       ...       ...       ...   ...       ...   
6035    -0.645455   0.0  0.000000  0.000000  0.000000   0.0  1.354545   
6036    -0.306333   0.0  0.693667 -0.306333  0.693667   0.0 -0.306333   
6037     0.000000   0.0  0.000000  0.000000  0.000000   0.0 -0.736196   
6039     0.000000   0.0  0.000000  0.000000  0.000000   0.0  0.000000   
6040     0.000000   0.0  0.000000  0.447876 -0.552124   0.0  0.447876   

MovieID      34    35        36        39   

## 2.1  User-based Colaborative Filtering

In [ ]:
user_sim_matrix = cosine_similarity(train_matrix_norm)
user_sim_df = pd.DataFrame(
    user_sim_matrix,
    index=train_matrix_norm.index,
    columns=train_matrix_norm.index
)

def predict_user_based(user_id, movie_id, n_neighbors=20):
    """
    Predict rating for (user_id, movie_id) using top-N similar users.
    Returns NaN if prediction cannot be made.
    """
    if user_id not in user_sim_df.index or movie_id not in train_matrix.columns:
        return np.nan

    # Users who rated this movie
    rated_mask = train_matrix[movie_id].notna()
    rated_users = train_matrix[movie_id][rated_mask].index

    if len(rated_users) == 0:
        return np.nan

    # Similarities to those users
    sims = user_sim_df.loc[user_id, rated_users]
    top_sims = sims.nlargest(n_neighbors)

    # Weighted average of mean-centred ratings + target user mean
    numerator   = (top_sims * train_matrix_norm.loc[top_sims.index, movie_id]).sum()
    denominator = top_sims.abs().sum()

    if denominator == 0:
        return user_means.get(user_id, 3.0)

    return user_means.get(user_id, 3.0) + numerator / denominator

In [ ]:
#user_sim_df

In [ ]:
# ── Evaluate on test set (sample 2 000 rows for speed) ───────────────────────
test_sample = test_df.sample(min(2000, len(test_df)), random_state=42)

ubcf_preds, ubcf_actuals = [], []
for _, row in test_sample.iterrows():
    pred = predict_user_based(row['UserID'], row['MovieID'])
    if not np.isnan(pred):
        ubcf_preds.append(np.clip(pred, 1, 5))
        ubcf_actuals.append(row['Rating'])

ubcf_rmse = np.sqrt(np.mean((np.array(ubcf_preds) - np.array(ubcf_actuals))**2))
ubcf_mae  = np.mean(np.abs(np.array(ubcf_preds) - np.array(ubcf_actuals)))
print(f"User-Based CF  →  RMSE: {ubcf_rmse:.4f}  |  MAE: {ubcf_mae:.4f}  "
      f"(coverage: {len(ubcf_preds)/len(test_sample)*100:.1f}%)")
#print(len(ubcf_preds),len(ubcf_actuals))

User-Based CF  →  RMSE: 0.9005  |  MAE: 0.7012  (coverage: 100.0%)


## 2.2  Item-based Colaborative Filtering

In [ ]:
item_sim_matrix = cosine_similarity(train_matrix_norm.T)
item_sim_df = pd.DataFrame(
    item_sim_matrix,
    index=train_matrix_norm.columns,
    columns=train_matrix_norm.columns
)

def predict_item_based(user_id, movie_id, n_neighbors=20):
    """
    Predict rating for (user_id, movie_id) using top-N similar items.
    """
    if user_id not in train_matrix.index or movie_id not in item_sim_df.index:
        return np.nan

    # Movies already rated by this user
    user_ratings = train_matrix.loc[user_id].dropna()
    if len(user_ratings) == 0:
        return np.nan

    sims = item_sim_df.loc[movie_id, user_ratings.index]
    top_sims = sims.nlargest(n_neighbors)

    numerator   = (top_sims * user_ratings[top_sims.index]).sum()
    denominator = top_sims.abs().sum()

    if denominator == 0:
        return user_means.get(user_id, 3.0)

    return numerator / denominator

In [ ]:
#item_sim_df

In [ ]:
# ── Evaluate ──────────────────────────────────────────────────────────────────
ibcf_preds, ibcf_actuals = [], []
for _, row in test_sample.iterrows():
    pred = predict_item_based(row['UserID'], row['MovieID'])
    if not np.isnan(pred):
        ibcf_preds.append(np.clip(pred, 1, 5))
        ibcf_actuals.append(row['Rating'])

ibcf_rmse = np.sqrt(np.mean((np.array(ibcf_preds) - np.array(ibcf_actuals))**2))
ibcf_mae  = np.mean(np.abs(np.array(ibcf_preds) - np.array(ibcf_actuals)))
print(f"Item-Based CF  →  RMSE: {ibcf_rmse:.4f}  |  MAE: {ibcf_mae:.4f}  "
      f"(coverage: {len(ibcf_preds)/len(test_sample)*100:.1f}%)")

Item-Based CF  →  RMSE: 0.8932  |  MAE: 0.6986  (coverage: 100.0%)


# 3. Evaluation

In [ ]:
RE_EVAL = False

In [ ]:
RELEVANCE_THRESHOLD = 4.0   # ratings ≥ 4 considered "relevant"
K_VALUES = [5, 10, 20]
all_movies = train_matrix.columns.tolist()

def dcg(scores):
    return sum(s / np.log2(i + 2) for i, s in enumerate(scores))

def ndcg_at_k(recommended, relevant_set, k):
    hits = [1 if m in relevant_set else 0 for m in recommended[:k]]
    ideal = sorted(hits, reverse=True)
    return dcg(hits) / dcg(ideal) if dcg(ideal) > 0 else 0.0

def ranking_metrics(pred_func, test_data, k_values, n_users=200):
    """Compute Precision@K, Recall@K, NDCG@K averaged over n_users."""
    results = {k: {'precision': [], 'recall': [], 'ndcg': []} for k in k_values}

    # Users present in both train and test
    test_users = test_data['UserID'].unique()
    train_users = set(train_matrix.index)
    eval_users = list(set(test_users) & train_users)[:n_users]

    for user_id in eval_users:
        # Ground truth relevant movies (from test set)
        user_test = test_data[test_data['UserID'] == user_id]
        relevant = set(user_test[user_test['Rating'] >= RELEVANCE_THRESHOLD]['MovieID'])
        if len(relevant) == 0:
            continue

        # Candidate movies: not seen in training
        seen = set(train_matrix.loc[user_id].dropna().index) if user_id in train_matrix.index else set()
        candidates = [m for m in all_movies if m not in seen]

        # Score candidates
        scores = [(m, pred_func(user_id, m)) for m in candidates]
        scores = [(m, s) for m, s in scores if not np.isnan(s)]
        scores.sort(key=lambda x: x[1], reverse=True)
        ranked = [m for m, _ in scores]

        for k in k_values:
            top_k = set(ranked[:k])
            hits  = top_k & relevant
            results[k]['precision'].append(len(hits) / k)
            results[k]['recall'].append(len(hits) / len(relevant))
            results[k]['ndcg'].append(ndcg_at_k(ranked, relevant, k))

    return {str(k): {m: np.mean(v) for m, v in metrics.items()} for k, metrics in results.items()}

if RE_EVAL:
    print("Computing ranking metrics for User-Based CF …")
    ubcf_rank = ranking_metrics(predict_user_based, test_df, K_VALUES, n_users=150)

    print("Computing ranking metrics for Item-Based CF …")
    ibcf_rank = ranking_metrics(predict_item_based, test_df, K_VALUES, n_users=150)
else:
    import json
    gdown.download('https://drive.google.com/uc?id=16mzMbVtWtjHpZ5PT1_3t-wyO5_O39P3R', output="./ubcf_rank.json", quiet=False)
    gdown.download('https://drive.google.com/uc?id=1nePvqz0JOnSSwTyQTzVWxJv5qB0WvYvW', output="./ibcf_rank.json", quiet=False)
    with open("./ubcf_rank.json", "r") as f:
        ubcf_rank = json.load(f)
    with open("./ibcf_rank.json", "r") as f:
        ibcf_rank = json.load(f)
    print("Loaded ranking metrics from file.")

Computing ranking metrics for User-Based CF …
Computing ranking metrics for Item-Based CF …


In [ ]:
# import json
# with open("ubcf_rank.json", "w") as f:
#     json.dump(ubcf_rank, f, indent=4)

# with open("ibcf_rank.json", "w") as f:
#     json.dump(ibcf_rank, f, indent=4)

In [ ]:
print(f"\n{'Model':<16}", end='')
for k in K_VALUES:
    print(f"  P@{k}   R@{k}  NDCG@{k}", end='')
print()
print("-" * 80)

for name, metrics in [("UserBased CF", ubcf_rank),
                       ("ItemBased CF", ibcf_rank)]:
    print(f"{name:<16}", end='')
    for k in K_VALUES:
        m = metrics[f'{k}']
        print(f"  {m['precision']:.4f} {m['recall']:.4f} {m['ndcg']:.4f}", end='')
    print()


Model             P@5   R@5  NDCG@5  P@10   R@10  NDCG@10  P@20   R@20  NDCG@20
--------------------------------------------------------------------------------
UserBased CF      0.1160 0.0229 0.2442  0.1027 0.0428 0.2971  0.0820 0.0693 0.3289
ItemBased CF      0.1067 0.0188 0.2232  0.0967 0.0344 0.2566  0.0790 0.0594 0.2833


In [ ]:
results_rows = []
for name, rank_m, rmse, mae in [
    ("User-Based CF", ubcf_rank, ubcf_rmse, ubcf_mae),
    ("Item-Based CF", ibcf_rank, ibcf_rmse, ibcf_mae),
]:
    row = {'Model': name, 'RMSE': round(rmse, 4), 'MAE': round(mae, 4)}
    for k in K_VALUES:
        row[f'P@{k}']    = round(rank_m[f'{k}']['precision'], 4)
        row[f'R@{k}']    = round(rank_m[f'{k}']['recall'],    4)
        row[f'NDCG@{k}'] = round(rank_m[f'{k}']['ndcg'],      4)
    results_rows.append(row)

results_df = pd.DataFrame(results_rows).set_index('Model')
print("\nConsolidated Results:\n", results_df.to_string())


Consolidated Results:
                  RMSE     MAE     P@5     R@5  NDCG@5    P@10    R@10  NDCG@10   P@20    R@20  NDCG@20
Model                                                                                                 
User-Based CF  0.9005  0.7012  0.1160  0.0229  0.2442  0.1027  0.0428   0.2971  0.082  0.0693   0.3289
Item-Based CF  0.8932  0.6986  0.1067  0.0188  0.2232  0.0967  0.0344   0.2566  0.079  0.0594   0.2833


In [ ]:
# ── (a) RMSE / MAE bar chart ──────────────────────────────────────────────────
fig_err = go.Figure()
models = ["User-Based CF", "Item-Based CF", "SVD"]
rmse_vals = [ubcf_rmse, ibcf_rmse]
mae_vals  = [ubcf_mae,  ibcf_mae]

fig_err.add_trace(go.Bar(name='RMSE', x=models, y=rmse_vals, marker_color='steelblue'))
fig_err.add_trace(go.Bar(name='MAE',  x=models, y=mae_vals,  marker_color='coral'))
fig_err.update_layout(
    title='Prediction Error: RMSE & MAE by Model',
    barmode='group', yaxis_title='Error', template='plotly_white'
)
fig_err.show()

In [ ]:
# ── (b) Precision@K, Recall@K, NDCG@K line charts ────────────────────────────
metrics_data = {
    "Precision@K": {name: [m[f'{k}']['precision'] for k in K_VALUES]
                    for name, m in [("User-Based CF", ubcf_rank),
                                    ("Item-Based CF", ibcf_rank)]},
    "Recall@K":    {name: [m[f'{k}']['recall']    for k in K_VALUES]
                    for name, m in [("User-Based CF", ubcf_rank),
                                    ("Item-Based CF", ibcf_rank)]},
    "NDCG@K":      {name: [m[f'{k}']['ndcg']      for k in K_VALUES]
                    for name, m in [("User-Based CF", ubcf_rank),
                                    ("Item-Based CF", ibcf_rank)]},
}

fig_rank = make_subplots(rows=1, cols=3,
                          subplot_titles=list(metrics_data.keys()))
colors = {'User-Based CF': 'steelblue', 'Item-Based CF': 'coral'}

for col_i, (metric_name, model_dict) in enumerate(metrics_data.items(), start=1):
    for model_name, vals in model_dict.items():
        fig_rank.add_trace(
            go.Scatter(x=K_VALUES, y=vals, mode='lines+markers',
                       name=model_name, showlegend=(col_i == 1),
                       marker_color=colors[model_name]),
            row=1, col=col_i
        )
    fig_rank.update_xaxes(title_text='K', row=1, col=col_i)
    fig_rank.update_yaxes(title_text=metric_name, row=1, col=col_i)

fig_rank.update_layout(title='Ranking Metrics @K by Model',
                        template='plotly_white', height=400)
fig_rank.show()

In [ ]:
# ── (c) User similarity heatmap (top 50 most active users) ───────────────────
top50 = user_counts.nlargest(50).index
sim_50 = user_sim_df.loc[top50, top50]

fig_heat = px.imshow(
    sim_50,
    title='User-User Cosine Similarity (Top 50 Most Active Users)',
    color_continuous_scale='RdBu', zmin=-1, zmax=1,
    labels=dict(color='Cosine Sim')
)
fig_heat.update_layout(template='plotly_white')
fig_heat.show()

In [ ]:
# ── (e) Rating error distribution: User-Based CF vs Item-Based CF ─────────────
ubcf_errors = np.array(ubcf_preds) - np.array(ubcf_actuals)
ibcf_errors = np.array(ibcf_preds) - np.array(ibcf_actuals)

fig_err_dist = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'User-Based CF',
        'Item-Based CF'
    ]
)
fig_err_dist.add_trace(
    go.Histogram(x=ubcf_errors, nbinsx=40, name='User-Based CF',
                 marker_color='steelblue', opacity=0.8),
    row=1, col=1
)
fig_err_dist.add_trace(
    go.Histogram(x=ibcf_errors, nbinsx=40, name='Item-Based CF',
                 marker_color='coral', opacity=0.8),
    row=1, col=2
)
fig_err_dist.update_xaxes(title_text='Predicted − Actual', row=1, col=1)
fig_err_dist.update_xaxes(title_text='Predicted − Actual', row=1, col=2)
fig_err_dist.update_yaxes(title_text='Count', row=1, col=1)
fig_err_dist.update_layout(
    title='Prediction Error Distributions: User-Based CF vs Item-Based CF',
    template='plotly_white', height=400, showlegend=True
)
fig_err_dist.show()

In [ ]:
# ── (f) Top-N similar users / items per model (replaces SVD factor analysis) ──
AGE_MAP = {1: 'Under 18', 18: '18-24', 25: '25-34', 35: '35-44',
           45: '45-49', 50: '50-55', 56: '56+'}
OCC_MAP = {
    0: 'other', 1: 'academic/educator', 2: 'artist', 3: 'clerical/admin',
    4: 'college/grad student', 5: 'customer service', 6: 'doctor/health care',
    7: 'executive/managerial', 8: 'farmer', 9: 'homemaker', 10: 'K-12 student',
    11: 'lawyer', 12: 'programmer', 13: 'retired', 14: 'sales/marketing',
    15: 'scientist', 16: 'self-employed', 17: 'technician/engineer',
    18: 'tradesman/craftsman', 19: 'unemployed', 20: 'writer'
}

# User-Based CF: show the 10 users most similar to the most active user
top_active_user = user_counts.idxmax()
top_similar_users = (
    user_sim_df.loc[top_active_user]
    .drop(index=top_active_user)
    .nlargest(10)
    .reset_index()
)
top_similar_users.columns = ['UserID', 'CosineSimilarity']
top_similar_users['CosineSimilarity'] = top_similar_users['CosineSimilarity'].round(4)

# Enrich with user demographic features
top_similar_users = top_similar_users.merge(
    users[['UserID', 'Gender', 'Age', 'Occupation']], on='UserID', how='left'
)
top_similar_users['Age']        = top_similar_users['Age'].map(AGE_MAP)
top_similar_users['Occupation'] = top_similar_users['Occupation'].map(OCC_MAP)
top_similar_users['Gender']     = top_similar_users['Gender'].map({'M': 'Male', 'F': 'Female'})

# Also show the anchor user's own profile for comparison
anchor_profile = users[users['UserID'] == top_active_user][['UserID', 'Gender', 'Age', 'Occupation']].copy()
anchor_profile['Age']        = anchor_profile['Age'].map(AGE_MAP)
anchor_profile['Occupation'] = anchor_profile['Occupation'].map(OCC_MAP)
anchor_profile['Gender']     = anchor_profile['Gender'].map({'M': 'Male', 'F': 'Female'})
anchor_profile['CosineSimilarity'] = 1.0  # self-similarity

print(f"\nAnchor user profile (User {top_active_user}):")
print(anchor_profile[['UserID', 'Gender', 'Age', 'Occupation']].to_string(index=False))

print(f"\nTop 10 users most similar to User {top_active_user} (User-Based CF):")
print(top_similar_users[['UserID', 'CosineSimilarity', 'Gender', 'Age', 'Occupation']].to_string(index=False))

# Item-Based CF: for each of the 3 most-rated movies, show top 5 similar items
top3_movies = movie_counts.nlargest(3).index.tolist()
print("\nTop 5 similar movies per most-rated movie (Item-Based CF):")
for mid in top3_movies:
    anchor_row = movies.loc[movies['MovieID'] == mid, ['Title', 'Genres']]
    title_str  = anchor_row['Title'].values[0]  if len(anchor_row) > 0 else str(mid)
    genres_str = anchor_row['Genres'].values[0] if len(anchor_row) > 0 else '—'

    similar = (
        item_sim_df.loc[mid]
        .drop(index=mid)
        .nlargest(5)
        .reset_index()
    )
    similar.columns = ['MovieID', 'CosineSimilarity']
    similar = similar.merge(movies[['MovieID', 'Title', 'Genres']], on='MovieID')
    similar['CosineSimilarity'] = similar['CosineSimilarity'].round(4)

    print(f"\n  Similar to '{title_str}'  [{genres_str}]:")
    print(similar[['Title', 'Genres', 'CosineSimilarity']].to_string(index=False))


Anchor user profile (User 4169):
 UserID Gender   Age Occupation
   4169   Male 50-55      other

Top 10 users most similar to User 4169 (User-Based CF):
 UserID  CosineSimilarity Gender   Age           Occupation
   1980            0.2919   Male 35-44 executive/managerial
   5795            0.2519   Male 25-34    academic/educator
    424            0.2488   Male 25-34  technician/engineer
   5831            0.2422   Male 25-34    academic/educator
   4344            0.2421   Male 25-34    academic/educator
   4064            0.2421   Male 45-49                other
   4543            0.2383   Male 25-34               artist
   4227            0.2318   Male 25-34           unemployed
   4447            0.2294   Male 45-49     customer service
   4277            0.2248   Male 35-44        self-employed

Top 5 similar movies per most-rated movie (Item-Based CF):

  Similar to 'American Beauty (1999)'  [Comedy|Drama]:
                           Title                           Genres  Co

In [ ]:
# Demo
def get_top_n_recommendations(user_id, model='user', n=10):
    """
    Return top-N unseen movie recommendations for a given user.
    model ∈ {'user', 'item'}
    """
    if user_id not in train_matrix.index:
        print(f"User {user_id} not found in training data.")
        return pd.DataFrame()

    seen = set(train_matrix.loc[user_id].dropna().index)
    candidates = [m for m in all_movies if m not in seen]

    pred_fn = predict_user_based if model == 'user' else predict_item_based

    scores = [(m, pred_fn(user_id, m)) for m in candidates]
    scores = [(m, s) for m, s in scores if not np.isnan(s)]
    scores.sort(key=lambda x: x[1], reverse=True)

    top_movies = pd.DataFrame(scores[:n], columns=['MovieID', 'PredictedRating'])
    top_movies = top_movies.merge(
        movies[['MovieID', 'Title', 'Genres']], on='MovieID'
    )
    top_movies['PredictedRating'] = top_movies['PredictedRating'].clip(1, 5).round(3)
    return top_movies[['Title', 'Genres', 'PredictedRating']]

# ── Demo for user 1 ───────────────────────────────────────────────────────────
DEMO_USER = 1
print(f"\n── User {DEMO_USER}: ratings history ──────────────────────────────")
user_history = train_df[train_df['UserID'] == DEMO_USER].sort_values('Rating', ascending=False)
print(user_history[['Title', 'Genres', 'Rating']].head(10).to_string(index=False))

for model_name in ['user', 'item']:
    label = 'User-Based CF' if model_name == 'user' else 'Item-Based CF'
    print(f"\n── Top 10 recommendations via {label} ──")
    recs = get_top_n_recommendations(DEMO_USER, model=model_name, n=10)
    print(recs.to_string(index=False))

# ── Visualise: compare User-Based vs Item-Based recommendations ───────────────
recs_user = get_top_n_recommendations(DEMO_USER, model='user', n=15)
recs_item = get_top_n_recommendations(DEMO_USER, model='item', n=15)

fig_recs = make_subplots(
    rows=2, cols=1,
    subplot_titles=[
        f'Top 15 User-Based CF Recs for User {DEMO_USER}',
        f'Top 15 Item-Based CF Recs for User {DEMO_USER}'
    ]
)
fig_recs.add_trace(
    go.Bar(
        x=recs_user.sort_values('PredictedRating')['PredictedRating'],
        y=recs_user.sort_values('PredictedRating')['Title'],
        orientation='h', name='User-Based CF',
        marker=dict(color=recs_user.sort_values('PredictedRating')['PredictedRating'],
                    colorscale='Blues', showscale=False)
    ),
    row=1, col=1
)
fig_recs.add_trace(
    go.Bar(
        x=recs_item.sort_values('PredictedRating')['PredictedRating'],
        y=recs_item.sort_values('PredictedRating')['Title'],
        orientation='h', name='Item-Based CF',
        marker=dict(color=recs_item.sort_values('PredictedRating')['PredictedRating'],
                    colorscale='Oranges', showscale=False)
    ),
    row=2, col=1
)
fig_recs.update_xaxes(title_text='Predicted Rating', row=1, col=1)
fig_recs.update_xaxes(title_text='Predicted Rating', row=2, col=1)
fig_recs.update_layout(
    title=f'Recommendation Comparison for User {DEMO_USER}: User-Based CF vs Item-Based CF',
    template='plotly_white', height=550, showlegend=False
)
fig_recs.show()


── User 1: ratings history ──────────────────────────────
                      Title                               Genres  Rating
             Ben-Hur (1959)               Action|Adventure|Drama       5
           Apollo 13 (1995)                                Drama       5
            Rain Man (1988)                                Drama       5
       Bug's Life, A (1998)          Animation|Children's|Comedy       5
Beauty and the Beast (1991)         Animation|Children's|Musical       5
        Mary Poppins (1964)            Children's|Comedy|Musical       5
 Saving Private Ryan (1998)                     Action|Drama|War       5
           Toy Story (1995)          Animation|Children's|Comedy       5
          Pocahontas (1995) Animation|Children's|Musical|Romance       5
               Dumbo (1941)         Animation|Children's|Musical       5

── Top 10 recommendations via User-Based CF ──
                                                              Title             Genres  Pr